# Lab 06 — VAE, GAN and diffusion intuition

**Dataset:** [ylecun/mnist](https://huggingface.co/datasets/ylecun/mnist). This laboratory keeps the models deliberately small so the learner can inspect the objective rather than wait for a large generative run.

We implement a VAE and a simple denoising diffusion process. GAN training is studied through the discriminator/generator loss curves and a controlled instability intervention.

## The math behind the experiment

A VAE samples a latent variable using

$$
z=\mu_\phi(x)+\sigma_\phi(x)\odot\epsilon,qquad \epsilon\sim\mathcal{N}(0,I).
$$

Its objective combines reconstruction with latent regularization. In this lab, reconstruction error is summed over pixels for each example, the KL divergence is summed over latent coordinates, and both are averaged over the batch:

$$
L_{VAE}=L_{rec}+\beta D_{KL}\left(q_\phi(z\mid x)\|p(z)\right).
$$

For diffusion, the forward process gradually mixes signal with noise. A useful intuition is that increasing the noise level $t$ decreases the amount of recoverable structure. The experiment measures that degradation rather than asking you to trust a generated picture.

In [ ]:
!pip -q install datasets torch pandas matplotlib
from datasets import load_dataset
import numpy as np,pandas as pd,torch
from torch import nn
tr=load_dataset("ylecun/mnist",split="train[:2000]")
X=torch.tensor(np.array([np.array(x).reshape(-1) for x in tr["image"]])/255.,dtype=torch.float32)
print(X.shape)

## Step 1 — Prediction

Predict that the VAE reconstruction error will decrease during training, while its KL term prevents the latent distribution from becoming an unconstrained lookup table. For diffusion, predict that repeated noise addition destroys recognizable structure while a trained denoiser can learn the reverse direction.

In [ ]:
class VAE(nn.Module):
 def __init__(self,z=8):
  super().__init__(); self.e=nn.Sequential(nn.Linear(784,128),nn.ReLU()); self.mu=nn.Linear(128,z); self.lv=nn.Linear(128,z); self.d=nn.Sequential(nn.Linear(z,128),nn.ReLU(),nn.Linear(128,784),nn.Sigmoid())
 def forward(self,x):
  h=self.e(x); mu,lv=self.mu(h),self.lv(h); z=mu+torch.exp(.5*lv)*torch.randn_like(mu); return self.d(z),mu,lv
m=VAE(); opt=torch.optim.Adam(m.parameters(),1e-3); rows=[]
beta=0.01  # Explicit beta-VAE trade-off; tune only with a held-out validation protocol in a larger study.
for ep in range(8):
 opt.zero_grad()
 rec,mu,lv=m(X)
 # Sum pixel reconstruction terms and latent KL terms per example, then average over the batch.
 r=((rec-X)**2).sum(dim=1).mean()
 kl=-.5*(1+lv-mu.pow(2)-lv.exp()).sum(dim=1).mean()
 loss=r+beta*kl
 loss.backward(); opt.step()
 rows.append([ep+1,r.item(),kl.item(),loss.item()])
vae_curve=pd.DataFrame(rows,columns=["epoch","reconstruction","KL","total"]); vae_curve

## Step 2 — Diffusion forward process

No training is needed to see the forward process: progressively mix Gaussian noise with an image. We average the corruption error over repeated noise draws so one lucky noise sample does not make the curve appear non-monotonic.

In [ ]:
x=X[0:1]; rows=[]
for t in [0.1,0.3,0.5,0.8,1.0]:
    # Simple Gaussian forward-diffusion step with alpha_bar decreasing over time.
    alpha_bar=(1-t)**2
    mse_values=[]; sample=None
    for _ in range(100):
        eps=torch.randn_like(x)
        sample=alpha_bar**0.5*x+(1-alpha_bar)**0.5*eps
        mse_values.append(((sample-x)**2).mean().item())
    rows.append([t,float(np.mean(mse_values)),float(np.std(mse_values)),sample.mean().item(),sample.std().item()])
diffusion_results=pd.DataFrame(rows,columns=["noise_fraction","MSE_from_original","MSE_sd","sample_mean","sample_std"])
diffusion_results

## Step 3 — GAN failure experiment

The key lesson is the adversarial objective, not a visually impressive sample. Use a toy one-dimensional distribution so mode collapse is visible as missing modes.

In [ ]:
torch.manual_seed(0)
real=torch.cat([torch.randn(500,1)-2,torch.randn(500,1)+2])
G=nn.Sequential(nn.Linear(1,16),nn.ReLU(),nn.Linear(16,1)); D=nn.Sequential(nn.Linear(1,16),nn.ReLU(),nn.Linear(16,1),nn.Sigmoid())
go=torch.optim.Adam(G.parameters(),1e-3); do=torch.optim.Adam(D.parameters(),1e-3); bce=nn.BCELoss(); history=[]
import time
gan_start=time.perf_counter()
for step in range(300):
 z=torch.randn(128,1); fake=G(z).detach(); do.zero_grad(); ld=bce(D(real[torch.randint(len(real),(128,))]),torch.ones(128,1))+bce(D(fake),torch.zeros(128,1)); ld.backward(); do.step()
 z=torch.randn(128,1); go.zero_grad(); lg=bce(D(G(z)),torch.ones(128,1)); lg.backward(); go.step()
 if step%50==0: history.append([step,ld.item(),lg.item(),G(torch.randn(1000,1)).std().item()])
gan_training_seconds=time.perf_counter()-gan_start
print(f"GAN training time for 300 updates: {gan_training_seconds:.2f} seconds (environment-specific)")
pd.DataFrame(history,columns=["step","D_loss","G_loss","generated_std"])

## Visualize generation — objectives and failure modes

The first graph tracks reconstruction and KL regularization through VAE training. The second shows how the diffusion forward process progressively changes an example. The last graph checks a GAN failure mode: generated samples can lose diversity even when loss curves look plausible. The final toy-distribution check reports per-mode coverage, generated spread, and the fixed training budget.

These are diagnostic plots, not a complete evaluation of sample quality.

In [ ]:
import matplotlib.pyplot as plt
# Compute the empirical 1-D Wasserstein-1 distance directly with NumPy; SciPy is not required.

fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))
axes[0].plot(vae_curve["epoch"], vae_curve["reconstruction"], marker="o", label="Reconstruction")
axes[0].plot(vae_curve["epoch"], vae_curve["KL"], marker="s", label="KL")
axes[0].plot(vae_curve["epoch"], vae_curve["total"], marker="^", label="Total")
axes[0].set_title("VAE objective terms")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Loss term")
axes[0].legend(fontsize=8)
axes[0].grid(alpha=.25)

diffusion_results = pd.DataFrame(rows, columns=["noise_fraction", "MSE_from_original", "MSE_sd", "sample_mean", "sample_std"])
axes[1].errorbar(diffusion_results["noise_fraction"], diffusion_results["MSE_from_original"], yerr=diffusion_results["MSE_sd"], marker="o", capsize=3)
axes[1].set_title("Forward diffusion: corruption")
axes[1].set_xlabel("Noise fraction")
axes[1].set_ylabel("MSE from original")
axes[1].grid(alpha=.25)

gan_history = pd.DataFrame(history, columns=["step", "D_loss", "G_loss", "generated_std"])
axes[2].plot(gan_history["step"], gan_history["generated_std"], marker="o")
axes[2].set_title("GAN: generated spread")
axes[2].set_xlabel("Training step")
axes[2].set_ylabel("Generated standard deviation")
axes[2].grid(alpha=.25)
fig.tight_layout()
plt.show()

# Evaluate against a fresh sample from the known toy distribution, not the examples used to train D.
torch.manual_seed(777)
real_eval=torch.cat([torch.randn(500,1)-2,torch.randn(500,1)+2])
with torch.no_grad():
    generated = G(torch.randn(2000, 1)).cpu().numpy().ravel()
plt.figure(figsize=(7, 4))
plt.hist(real_eval.cpu().numpy().ravel(), bins=40, density=True, alpha=.5, label="Fresh real sample")
plt.hist(generated, bins=40, density=True, alpha=.5, label="Generated")
plt.title("GAN mode coverage on a two-mode toy distribution")
plt.xlabel("Value")
plt.ylabel("Density")
plt.legend()
plt.tight_layout()
plt.show()

# Transparent toy metrics: count generated values within +/-1 of each real mode center.
left_fraction=float(np.mean(np.abs(generated+2)<1))
right_fraction=float(np.mean(np.abs(generated-2)<1))
mode_coverage=sum(fraction>=.05 for fraction in [left_fraction,right_fraction])/2
def empirical_wasserstein_1d(samples_a, samples_b):
    a=np.sort(np.asarray(samples_a).ravel())
    b=np.sort(np.asarray(samples_b).ravel())
    if a.size == 0 or b.size == 0:
        raise ValueError('Wasserstein distance requires non-empty samples')
    support=np.sort(np.concatenate([a,b]))
    cdf_a=np.searchsorted(a,support,side='right')/a.size
    cdf_b=np.searchsorted(b,support,side='right')/b.size
    return float(np.sum(np.abs(cdf_a[:-1]-cdf_b[:-1])*np.diff(support)))
real_samples=real_eval.cpu().numpy().ravel()
wasserstein_1=empirical_wasserstein_1d(real_samples,generated)
gan_metrics=pd.DataFrame([
    ["fresh real-sample standard deviation",float(real_eval.std().item())],
    ["generated standard deviation",float(np.std(generated))],
    ["generated fraction near mode -2",left_fraction],
    ["generated fraction near mode +2",right_fraction],
    ["two-mode coverage fraction",mode_coverage],
    ["empirical Wasserstein-1 distance (toy distribution)",wasserstein_1],
    ["training updates",300],
    ["training seconds (environment-specific)",gan_training_seconds]
],columns=["metric","value"])
display(gan_metrics)
print("Wasserstein-1 compares the two 1-D sample distributions; it is not an image-quality metric.")

plt.figure(figsize=(6,4))
plt.bar(["Mode near -2","Mode near +2"],[left_fraction,right_fraction])
plt.axhline(.05,linestyle="--",label="5% coverage threshold")
plt.ylim(0,1)
plt.ylabel("Fraction of generated samples")
plt.title("Toy GAN: explicit mode coverage")
plt.legend()
plt.tight_layout()
plt.show()

## Conclusion / answer key

**Expected:** VAE reconstruction improves while KL regularization remains nonzero; forward diffusion steadily increases corruption; GAN losses need not monotonically decrease and generator variance can collapse even when the discriminator/generator losses look superficially plausible. The toy mode-coverage table distinguishes spread from whether both target modes receive at least 5% of generated samples; this threshold is illustrative, not a standard benchmark.

### Research extension
Compare VAE latent dimension at fixed training steps and measure both reconstruction error and latent-class separability.